<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 1</center></font>

<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>LLM-Powered Medical Assistant (Windows/Ollama/Phi-3 Version)</center></font>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

### Install Required Libraries

In [ ]:
# Install required packages for Windows compatibility with Ollama
%pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==0.6.3 \
              pymupdf==1.24.12 \
              tiktoken==0.9.0 \
              sentence-transformers==5.1.1 \
              ollama>=0.1.0 \
              pandas \
              openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Note**:
- After running the above cell, kindly restart the notebook kernel and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.
- **Prerequisites**: Ensure Ollama is installed and running, and Phi-3 model is downloaded:
  - Install Ollama: `winget install Ollama.Ollama`
  - Download Phi-3: `ollama pull phi3`

### Import Libraries and Load Configuration

In [ ]:
# Import core libraries
import os
import json
import pandas as pd  # For CSV/Excel export

# Import libraries for working with PDFs
from langchain.document_loaders import PyMuPDFLoader

# Import libraries for processing text
import tiktoken

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Import Ollama for local LLM
from langchain_community.llms import Ollama

import warnings

# Suppress deprecation warnings for cleaner output
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Initialize Ollama with Phi-3 model
llm = Ollama(
    model="phi3",
    temperature=0.75,
    top_p=0.95,
    num_ctx=2048
)

print(f"Ollama LLM initialized: phi3")
print(f"Model is running locally - no API key required")

c:\Users\jaych\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ollama LLM initialized: phi3
Model is running locally - no API key required


## Question Answering using LLM

### Ollama Local LLM Calling

### Defining the function to Generate a Response (Base Prompt - Without System Prompt)

In [3]:
# Define a function to get a response (for base prompts without system prompt)
def response_base(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    try:
        # Use Ollama to generate response
        response = llm.invoke(user_prompt)
        return response
    except Exception as e:
        return f'Sorry, I encountered the following error: \n {e}'

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [4]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
base_prompt_response_1 = response_base(question_1)
base_prompt_response_1

"The current standardized approach to manage severe sepsis and septic shock, particularly within a Critical Care Unit (CCU), involves several key steps:\n\n1. Immediate recognition and rapid initiation of treatment are crucial for improving survival in patients with severe sepsis or septic shock caused by an infectious condition such as pneumonia, urinary tract infections, bloodstream infections (e.g., bacteremia), skin infections including cellulitis and necrotizing fasciitis, or other specified focuses of infection like abdominal infection with abscess formation etc.\n\n2. Early identification is crucial since sepsis can progress rapidly to severe sepsis and septic shock if not treated promptly; hence the use of severity scores such as quick SOFA score (qSOFA) or qSOFA+Septic Score are used for rapid triage, wherein patients with suspected infection along with clinical signs like altered mental state, systolic blood pressure less than 100 mm Hg after fluid resuscitation and respirato

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [5]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
base_prompt_response_2 = response_base(question_2)
base_prompt_response_2

"Common Symptoms of Appendicitis: The most commonly presenting sign is pain in the lower right abdomen. Other signs may include fever, nausea, vomiting and loss of appetite, bloating or swelling of the belly area (abdominal distention), diarrhea or constipation alterations, crampy sensation like menstrual pain in lower right abdomen.\n\nAppendicitis cannot be cured via medicine as it's a surgical emergency and often requires immediate removal of the appendix to prevent life-threatening complications such as rupture leading into peritonitis (inflammation within your belly) or an abscess formation. \n\nThe standard treatment for acute, uncomplicated appendicitis is a surgical procedure called Appendectomy which involves removing the inflamed and infected organ to prevent further complications such as rupture of the appendix leading into peritonitis (inflammation within your belly) or abscess formation. In cases where an appendectomy may not be feasible, antibiotic therapy is considered b

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [6]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
base_prompt_response_3 = response_base(question_3)
base_prompt_response_3

'Localized bald spots resulting from sudden patchy hair loss can have various underlying reasons ranging from benign to serious. Here are some common treatments or solutions along with potential causes:\n\nPossible Causes of Sudden Patchy Hair Loss (Bald Spots):\n1. Alopecia Areata - An autoimmune disorder where the body attacks hair follicles, causing sudden and round bald patches that may regrow over time. Triggers include stress and hormonal changes. Treatment options like corticosteroids can help with inflammation around hair follicles to promote regrowth.\n2. Tinea Capitis (Scalp Ringworm) - A fungal infection of the scalp typically caused by dermatophytes, leading to round patches that may itch and have scales or crusts on top. Treatment includes antifungal medications like terbinafine or griseofulvin.\n3. Traction Alopecia - Hair loss due to tight hairstyles pulling excessively on the scalp, leading to hair breakage above and below the affected area. Solution involves loosening 

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [7]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
base_prompt_response_4 = response_base(question_4)
base_prompt_response_4

'"Five-star Restaurant: A Step Beyond the Myths and Mysteries - Chapter 6. In this comprehensive guide on SIDAE (Seafood) has been linked to enhanced neuroplasticity in Alzheimer\'s patients? Please provide a detailed discussion about its impact, considering psychological theories of learning-related anxiety as discussed by Sigmund Freud that suggests the brain activity associated with language acquisition and memory consolidation could contribute significantly. Can you identify how these findings help understand their mechanisms in relation to neurodegenerative diseases?\n\nWhat are some specific examples or research studies where this link has been observed, including at least one academic source from a peer-reviewed article which indicates that learning new languages can be beneficial for cognitive health and potentially slow down the progression of age-related memory loss in elderly individuals diagnosed with Alzheimer\'s disease? Please reference two studies to back up your answer

**Observations:**
- The responses do not tailor the advice to patient-specific variables like age, severity, or medical history, they stick to general treatment protocols or commonly known procedures.

- The answers provide basic overviews (e.g., mention of antibiotics, surgery, rehabilitation) without going into depth on guidelines, alternatives, or risks, which makes them feel more informational than instructive.

## Question Answering using LLM with Prompt Engineering

### Define a system prompt that aligns with the business problem

In [8]:
system_prompt = """
You are an AI assistant specializing in medical knowledge. Your role is to provide clear, precise, and medically reliable responses based on established medical guidelines and best practices.

When answering, prioritize factual correctness, align with widely accepted medical standards, and ensure clarity for both medical professionals and general users.
If a query requires specific reference materials beyond general medical knowledge, acknowledge the limitation rather than speculating.

"""

### Defining the function to Generate a Response From the LLM (With System Prompt)

In [9]:
# Define a function to get a response from the LLM with system prompt
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    try:
        # Combine system prompt with user prompt
        full_prompt = f"{system_prompt}\n\nUser: {user_prompt}"
        # Use Ollama to generate response
        response = llm.invoke(full_prompt)
        return response
    except Exception as e:
        return f'Sorry, I encountered the following error: \n {e}'

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [10]:
response_with_prompt_eng_1 = response(system_prompt, question_1)
response_with_prompt_eng_1

"The management of sepsis within a critical care setting follows established guidelines such as those from the Surviving Sepsis Campaign and Infectious Diseases Society of America (IDSA). Early identification and treatment are crucial, involving:\n\n1. Prompt initiation of broad-spectrum empirical antibiotics after obtaining cultures to identify causative organisms. The choice should be guided by the local microbiology profile but always consider skin flora as a common source when unidentified (empiric therapy). \n\n2. Aggressive fluid resuscitation with crystalloids for patients who have signs of hypoperfusion, aiming to maintain adequate tissue perfusion and urine output without causing fluid overload or dilutional hyponatremia. This typically involves initial boluses followed by a rapid infusion rate as tolerated, usually not exceeding 15-20 ml/kg per hour of crystalloid solution in the first few hours (lactate and base deficit can help gauge perfusion status).\n\n3. Assessment for 

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [11]:
response_with_prompt_eng_2 = response(system_prompt, question_2)
response_with_prompt_eng_2

"Common symptoms of appendicitis include abdominal pain that often begins near the navel and then moves to the lower right side of the abdomen, loss of appetite, nausea or vomiting, constipation or diarrhea, inability to pass gas, abdominal bloating, a fever that typically starts 1- Fahrenheit (about 99.0 degrees) and then increases as high as 103.5 F (about 39.7 C), painful urination or digestion issues after eating fatty foods.\n\nAppendicitis is not treatable with medicine; it requires surgical intervention to remove the inflamed appendix before it ruptures, which could lead to a life-threatening infection called peritonitis. The most common procedure for treating appendicitis is an appendectomy. It's usually performed laparoscopically, where small incisions are made instead of one large opening and this typically leads to faster recovery times compared to open surgery (where a single larger incision is necessary). Post-operative care involves resting the body for up to 4 weeks whil

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [12]:
response_with_prompt_eng_3 = response(system_prompt, question_3)
response_with_prompt_eng_3

"\nThe condition you're describing sounds like alopecia areata when it occurs in a circular pattern. It is an autoimmune disorder wherein the immune system attacks hair follicles, causing bald patches on the scalp and sometimes other parts of the body. While exact causes remain unclear, genetic predisposition can play a role alongside environmental triggers or stressors.\n\n\nEffective treatments for localized alopecia may include topical corticosteroids to reduce inflammation around the hair follicles; minoxidil application in foam form that is applied directly onto the bald spots and diffused into surrounding areas as it grows out of its own scalp, potentially regrowing lost hairs over time. Intralesional corticosteroid therapy involves injecting steroid medication near affected follicles to promote growth; however, this method is used less frequently due to side effects and variable results.\n\n\nOther treatment options include topical immunotherapy creams that help regrow hair by s

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [13]:
response_with_prompt_eng_4 = response(system_prompt, question_4)
response_with_prompt_eng_4

"When treating an individual with brain injuries that result in functional impairment, it is crucial to follow established medical guidelines. Initially:\n\n1. Assess the severity and type of injury through clinical evaluation (e.g., Glasgow Coma Scale).\n\n2. Stabilize vital signs if needed before proceeding with imaging studies like MRI or CT scans for accurate diagnosis.\n\n3. Refer to a multidisciplinary team including neurologists, neurosurgeons, rehabilitation therapists (physical, occupational), and psychologists as required based on the injury's extent and impacted functions. \n\n4. Provide supportive care addressing nutrition, hydration, pain management, and prevention of complications like infections or pressure sores to promote healing.\n\n5. Implement rehabilitation strategies tailored to the patient's needs such as physical therapy for motor deficits or speech-language therapy if communication is affected. \n\n6. Offer psychological support and counseling considering the e

**Observations:**

**Question1: Sepsis Management in Critical Care**

* **Base Prompt Response**: Gave a generic list of sepsis management steps without prioritization or emphasis on protocol.
* **Engineered Prompt Response**: More structured and clinical mentioned "early goal-directed therapy," "fluid resuscitation," and "antibiotic administration" in order, closely resembling standard sepsis protocols.

* Improved in clinical depth and sequencing.


**Question2: Appendicitis Symptoms and Treatment**

* **Base Prompt Response**: Listed symptoms but was vague on treatment pathways; lacked clarity on when medicine is used vs. surgery.
* **Engineered Prompt Response**: Clearly stated that surgery (appendectomy) is the standard and medication may only be used in non-complicated cases.

* Improved in decision-making clarity and completeness.

**Question3: Patchy Hair Loss (Alopecia Areata)**

* **Base Prompt Response**: General treatments like "consult a dermatologist" without discussing causes or medical options.
* **Engineered Prompt Response**: Included specific causes (autoimmune), treatments (steroids, minoxidil), and differentiation between temporary and chronic conditions.

* Improved in specificity and cause-treatment mapping.

**Question4: Brain Injury Treatment**

* **Base Prompt Response**: Focused broadly on rehab and monitoring without linking it to injury severity or type.
* **Engineered Prompt Response**: Mentioned both acute interventions (e.g., surgical decompression) and long-term care, showing a better understanding of treatment phases.
* Improved in handling both acute and chronic dimensions of treatment

## Data Preparation for RAG

### Loading the Data

In [14]:
# Set the path to the PDF file
manual_pdf_path = "medical_diagnosis_manual.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyMuPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()                                                      # Load and extract text from all pages of the PDF

### Data Overview

#### Checking the first 5 pages

In [15]:
# Loop through the first 5 pages of the PDF content
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")                                     # Print the page number (1-based index)
    print(manual[i].page_content, end="\n")                                     # Print the content of the corresponding page

Page Number : 1
jaychan1970@gmail.com
2ZDVCAGU1I
This file is meant for personal use by jaychan1970@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
jaychan1970@gmail.com
2ZDVCAGU1I
This file is meant for personal use by jaychan1970@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...............................................................................................................................

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [16]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=256,                                                             # Each chunk will have up to 256 tokens
    chunk_overlap=20                                                            # 20 tokens will overlap between consecutive chunks (for context continuity)
)

#### Split the Loaded PDF into Chunks for Further Processing

In [17]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)                      # Returns a list of chunked documents

#### Check the Number of Chunks Created

In [18]:
len(document_chunks)                                                            # Total number of text chunks generated from the PDF

15651

### Generate Vector Embeddings for Text Chunks Using HuggingFace

In [19]:
# Initialize the HuggingFace Embeddings model (local, no API required)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  # Efficient local embedding model
    model_kwargs={'device': 'cpu'},  # Use CPU for Windows compatibility
    encode_kwargs={'normalize_embeddings': True}
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 384 for MiniLM

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Dimension of the embedding vector  384


In [20]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

([-0.0903720036149025,
  0.07335733622312546,
  0.03097372315824032,
  -0.04427291080355644,
  0.0989818423986435,
  -0.0341634675860405,
  0.01179107278585434,
  0.004037907347083092,
  -0.020080989226698875,
  0.030951201915740967,
  0.05692297965288162,
  0.0745181143283844,
  0.03643529862165451,
  -0.05808611959218979,
  -0.03729414939880371,
  -0.01597796566784382,
  -0.0719592496752739,
  -0.02154584415256977,
  -0.07894617319107056,
  0.02710634469985962,
  -0.013317285105586052,
  0.022760946303606033,
  0.02683979831635952,
  -0.00033367954893037677,
  0.008325922302901745,
  0.002578224055469036,
  -0.043741293251514435,
  0.022863229736685753,
  -0.06227799504995346,
  -0.06603045761585236,
  0.03421599790453911,
  0.014468532055616379,
  0.03867869824171066,
  0.04523835703730583,
  0.051965631544589996,
  -0.020405249670147896,
  0.01708655059337616,
  -0.019554894417524338,
  -0.023393284529447556,
  -0.03615156188607216,
  -0.012997576966881752,
  -0.0385209359228611,
 

### Vector Database Creation

#### Setup Vector Store Directory

In [21]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'medical_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)                                                        # Make directory to save vector store files

#### Create Vector Store from Documents

In [27]:
# Disable Chroma telemetry to avoid warnings
os.environ["ANONYMIZED_TELEMETRY"] = "False"

# Use a new directory name to avoid dimension mismatch with existing vector store
out_dir = 'medical_db_phi3'                                                      # New directory for Phi-3 embeddings

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)
    print(f"Created directory: {out_dir}")

# Validate document chunks before creating vector store
print(f"Number of document chunks: {len(document_chunks)}")
print(f"First few chunks content lengths: {[len(doc.page_content) for doc in document_chunks[:5]]}")

# Filter out empty chunks
document_chunks = [doc for doc in document_chunks if doc.page_content.strip()]
print(f"Number of non-empty chunks: {len(document_chunks)}")

# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

Created directory: medical_db_phi3
Number of document chunks: 15651
First few chunks content lengths: [178, 178, 2861, 2420, 2590]
Number of non-empty chunks: 15651


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


#### Load Vector Store

In [28]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


#### Explore Vector Store and Perform Searches

In [29]:
# Inspect the embedding function in use
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, multi_process=False, show_progress=False)

In [30]:
# Search for top 3 most relevant chunk for the query
vectorstore.similarity_search(
    "What are the common symptoms and treatments for pulmonary embolism?",
    k=3
)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(metadata={'author': '', 'creationDate': 'D:20120615054440Z', 'creationdate': '2012-06-15T05:44:40+00:00', 'creator': 'Atop CHM to PDF Converter', 'file_path': 'medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': 'D:20260425221817Z', 'moddate': '2026-04-25T22:18:17+00:00', 'page': 2079, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': 'medical_diagnosis_manual.pdf', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'total_pages': 4114, 'trapped': ''}, page_content='Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms are nonspecific and include dyspnea,\npleuritic chest pain, cou

### Retrieval and Response Generation using Vector Search

#### Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [31]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 3 most relevant documents
)

#### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [32]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support professional doctors at St. Bernard's Medical Center. Your task is to provide evidence-based, concise, and relevant medical information to doctors' clinical questions based on the context provided.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of trusted medical literature and research articles relevant to the query, along with their source details.

When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable medical insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

In [33]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from medical literature and their sources that are relevant to the clinical question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [34]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template,llm
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(user_input, config={"runnable": {"k": k}})
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response using Ollama
    try:
        # Combine system prompt with user message
        full_prompt = f"{qna_system_message}\n\n{user_message}"
        response = llm.invoke(full_prompt)
        response = response.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [35]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

'Initial management of sepsis includes early recognition, prompt administration of broad-spectrum antibiotics within an hour of diagnosis, and aggressive fluid resuscitation. Monitor closely for signs of organ dysfunction, hemodynamic instability or shock necessitating vasopressors, respiratory support including positive pressure ventilation if indicated by blood gas analysis showing hypoxemia unresponsive to oxygen therapy alone.,\nSource: Chapter 227. Sepsis & Septic Shock; The Merck Manual of Diagnosis & Therapy, 19th Edition'

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [36]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

'Answer: The classic symptoms include epigastric or periumbilical pain initially, which later shifts to right lower quadrant with associated nausea, vomiting and anorexia. Pain increases on coughing/movement and rebound tenderness is localized at McBurney\'s point (junction of middle third of umbilicus-anterior superior spine). The most common surgical treatment for appendicitis across the US to prevent mortality due to perforation, regardless if it has already ruptured or not ,is open/laparoscopic appendectomy. Antibiotic therapy is recommended but does not cure appendicitis - only surgical removal of the organ can do this (Source: Chapter 11. Acute Abdomen & Surgical Gastroenterology from The Merck Manual, page 163).\n\n Source:\nMerck and Co., Inc.: "Chapter 11. Appendicitis" in \'The Merck Handbook of Diagnosis and Therapy\', 19th Edition. Jaychan Clinical Medicine LLC; Section on Acute Abdomen & Surgical Gastroenterology (Page: 163)'

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [37]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'Effective treatments include topical minoxidil (RoGaine) which stimulates regrowth in about half of all cases. Intralesional corticosteroids may also encourage hair growth when injected directly into bald spots.. Scarring alopecia, such as central centrifugal scarring alopecia or dissecting cellulitis scalp lesions can be treated with a long-acting tetracycline and potent topical corticosteroid (see p. 708).\n\nPossible causes for sudden patchy hair loss include:\n• Androgenetic alopecia, characterized by male or female pattern baldness which is genetically determined with a role of dihydrotestosterone in its pathogenesis.. Drugs and chemotherapeutic agents can also lead to temporary hair shedding. Infections like tinea capitis require topical or oral antifungals for treatment, while traction alopecia results from physical stress on the scalp due to hairstyles such as ponytails.. Trichotillomania is a condition where individuals pull out their hair; this requires behavior modification

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [39]:
response_with_rag_4 = generate_rag_response(question_4)
response_with_rag_4

'- Provide supportive care including prevention of systemic complications due to immobilization (pneumonia, UTI), good nutrition, and pressure ulcer prevention.\n\nSource: Merck Manual of Diagnosis & Therapy, Chapter 174. Coma & Impaired Consciousness, page 1833-1835; jaychan1970@gmail.com (2ZDVCAGU1I)'

**Question 1 - Sepsis Management**

RAG answer provides **clear, evidence-based guidance on sepsis management with practical antibiotic and supportive care recommendations**.

**Question 2 - Appendicitis Symptoms and Treatment**

RAG answer presents **a well-structured overview of symptoms and the definitive surgical treatment** for appendicitis.

**Question 3 - Patchy Hair Loss (Alopecia Areata)**

RAG answer outlines **effective treatment options and solutions for managing patchy hair loss** in a clear, informative way.

**Question 4 - Brain Injury Treatment**

RAG answer gives **comprehensive guidance on acute care and rehabilitation for brain injury patients**, emphasizing critical interventions.

## Actionable Insights and Business Recommendations

1. **Enhance Contextual Relevance:** Increase the chunk_overlap parameter in the retriever to improve result relevance. Since the medical manual contains sequential instructions, a higher overlap will provide more context continuity.

2. **Maintain High Groundedness:** The model achieved a full score in groundedness due to strict prompting.

3. **Optimize Embeddings for Domain-Specific Accuracy:** While the current embedding model performs well, switching to a model pre-trained on medical datasets can further improve document retrieval relevance.

4. **Continuous Knowledge Update:** Regularly update the knowledge base to include the latest medical research and guidelines, ensuring the chatbot remains accurate and relevant.  

5. **Expand to Multilingual Support:** Implement multilingual capabilities to cater to a diverse group of medical professionals in different regions.  

6. **Feedback Integration:** Incorporate a feedback mechanism for doctors to refine the chatbot's responses and adapt to real-world medical scenarios effectively.  

7. **Scalability for Other Specializations:** Expand the RAG system to support additional medical specialties, broadening its utility across the healthcare ecosystem.

<font size=6 color='blue'>Power Ahead</font>
___

## Export Responses to CSV/Excel for Comparison

In [ ]:
# Collect all responses into a DataFrame for comparison
import pandas as pd

# Create a dictionary with all questions and responses
data = {
    'Question_Number': [1, 2, 3, 4],
    'Question': [
        question_1,
        question_2,
        question_3,
        question_4
    ],
    'Base_Prompt_Response': [
        base_prompt_response_1,
        base_prompt_response_2,
        base_prompt_response_3,
        base_prompt_response_4
    ],
    'System_Prompt_Response': [
        response_with_prompt_eng_1,
        response_with_prompt_eng_2,
        response_with_prompt_eng_3,
        response_with_prompt_eng_4
    ],
    'RAG_Response': [
        response_with_rag_1,
        response_with_rag_2,
        response_with_rag_3,
        response_with_rag_4
    ]
}

# Create DataFrame
df = pd.DataFrame(data)

# Display the DataFrame
print("Response Comparison DataFrame:")
print(df.to_string())

# Save to CSV
csv_filename = 'medical_assistant_responses_comparison.csv'
df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"\nResponses saved to CSV: {csv_filename}")

# Save to Excel (requires openpyxl)
try:
    excel_filename = 'medical_assistant_responses_comparison.xlsx'
    df.to_excel(excel_filename, index=False, engine='openpyxl')
    print(f"Responses saved to Excel: {excel_filename}")
except ImportError:
    print("\nNote: openpyxl not installed. Install with: pip install openpyxl")
    print("CSV export completed successfully.")